### Full Fine-Tuning: Your 100M Python-Code GPT on the New Dataset

This continues training your existing `nanoGPT`-style 100M model (the one from `base100m.ipynb`, already pushed to Hugging Face) on your **new problem/solution dataset** — the `text` / `problem` / `solution` / `tests` format you showed.

**Steps in this notebook:**
1. Install packages
2. Upload & load your dataset (JSON-lines file — one JSON object per line, matching the format you showed)
3. Tokenize the `text` field with the same DeepSeek-Coder tokenizer used for pretraining
4. Rebuild the exact model architecture and load your pretrained weights from Hugging Face
5. **Full-parameter** fine-tune — every weight is trainable, no LoRA/freezing
6. **Test**: generate code for held-out problems and run the real `assert` tests against it (pass@1)
7. Push the fine-tuned model to the Hugging Face Hub

Fill in the 4 values in the config cell before running.

In [ ]:
!pip install -q -U datasets transformers huggingface_hub safetensors tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 122.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 770.3/770.3 kB 57.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 676.6/676.6 kB 43.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 54.9 MB/s eta 0:00:00


### 1. Config — fill these in

In [ ]:
# ======================= FILL THESE IN =======================
DATASET_FILE = "dataset.jsonl"                  # your dataset file: one JSON object per line, must contain "text", "problem", "tests"
HF_BASE_REPO = "Ananda100/pocketcoder100M-distilled"    # the HF repo where your PRETRAINED model currently lives (from base100m.ipynb)
HF_FT_REPO   = "Ananda100/100m-sftd-python"      # where the FINE-TUNED model will be pushed
HF_TOKEN     = ""                                # your HF token (needs write access for the final push)
# ===============================================================
HF_DATASET_REPO = "Ananda100/python-sft-dataset"      # your private dataset repo
DATASET_SPLIT    = "train"

### 2. Load your dataset from the Hugging Face Hub

In [ ]:
from datasets import load_dataset

print(f"Loading dataset from Hugging Face Hub: {HF_DATASET_REPO} (split={DATASET_SPLIT})")
hf_ds = load_dataset(HF_DATASET_REPO, split=DATASET_SPLIT, token=HF_TOKEN or None)
print(hf_ds)
print("Columns:", hf_ds.column_names)

Loading dataset from Hugging Face Hub: Ananda100/python-sft-dataset (split=train)


README.md:   0%|          | 0.00/28.0 [00:00<?, ?B/s]

datasetsforsft.jsonl: reconstructing file:   0%|          |  0.00B /  936MB            

datasetsforsft.jsonl: downloading bytes:           |  0.00B            

Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['text', 'problem', 'solution', 'tests', 'source', 'layer', 'id'],
    num_rows: 280317
})
Columns: ['text', 'problem', 'solution', 'tests', 'source', 'layer', 'id']


### 3. Convert to records and split

Each row is a JSON object like:
```json
{"text": "### Problem\n...\n\n### Solution\n```python\n...\n```", "problem": "...", "solution": "...", "tests": "[\"assert ...\"]", "source": "...", "layer": "...", "id": 0}
```

- `problem` + `solution` are the ONLY fields used for training (built into the same `### Problem / ### Solution` template at tokenization time, in section 4) — the precomputed `text` column, `source`, and `layer` are not used at all.
- `tests` (+ `problem`) are held out for the **functional test** at the end (generate code, run the real asserts).
- `id` must be present and unique (your `build_sft_mix.ipynb` already writes this) — it's how the held-out test set is chosen without overlapping the training set. If the repo doesn't have one, this cell adds one automatically.
- Note: if your file came from `build_sft_mix.ipynb`, only the OpenCodeInstruct-sourced rows have a real `tests` value — every other source's `tests` is `null`. That's handled automatically below.

In [ ]:
import json
from datasets import Dataset

records = hf_ds.to_list()

# Make sure every record has a unique "id" - needed below to pick the held-out test
# set without overlapping train/val. Some HF dataset repos won't carry this column.
if not records or "id" not in records[0] or any(r.get("id") is None for r in records):
    for i, r in enumerate(records):
        r["id"] = i

print(f"Loaded {len(records)} examples")
print("Example keys:", list(records[0].keys()))

# Only OpenCodeInstruct records carry real unit tests (per build_sft_mix.ipynb) -
# every other source's "tests" field is null. Split those out separately so the
# functional test in section 11 only draws from examples that actually have asserts to run.
def has_tests(r):
    t = r.get("tests")
    return t not in (None, "", "null", [])

records_with_tests = [r for r in records if has_tests(r)]
print(f"Examples with usable tests: {len(records_with_tests)} / {len(records)}")

import random
random.seed(42)
n_test = min(200, len(records_with_tests))
if n_test == 0:
    raise ValueError("No examples with non-null \"tests\" found - the functional test cell needs at least a few.")
test_ids = set(r["id"] for r in random.sample(records_with_tests, n_test))

train_val_records = [r for r in records if r["id"] not in test_ids]
test_records = [r for r in records if r["id"] in test_ids]

# 98% train / 2% val (loss tracking) from everything NOT held out for the functional test
tv_ds = Dataset.from_list(train_val_records).train_test_split(test_size=0.02, seed=42)

ds = {
    "train": tv_ds["train"],
    "val":   tv_ds["test"],
    "test":  Dataset.from_list(test_records),  # kept RAW - used only in the functional test cell
}
print({k: len(v) for k, v in ds.items()})

Loaded 280317 examples
Example keys: ['text', 'problem', 'solution', 'tests', 'source', 'layer', 'id']
Examples with usable tests: 89211 / 280317
{'train': 274514, 'val': 5603, 'test': 200}


### 4. Tokenize — built ONLY from `problem` + `solution` (same tokenizer as pretraining)

In [ ]:
from transformers import AutoTokenizer
import numpy as np, os
from tqdm.auto import tqdm

tokenizer = AutoTokenizer.from_pretrained("deepseek-ai/deepseek-coder-6.7b-base", trust_remote_code=True)
vocab_size = len(tokenizer)
print(f"Vocab size: {vocab_size}")

if tokenizer.eos_token_id is None:
    raise ValueError("Tokenizer has no eos_token_id set - fix before tokenizing.")

# Training text is built ONLY from problem + solution - the precomputed "text" column,
# "tests", "source", and "layer" are ignored entirely for training.
def format_example(problem, solution):
    return f"### Problem\n{problem}\n\n### Solution\n```python\n{solution}\n```"

def process(example):
    text = format_example(example["problem"], example["solution"])
    ids = tokenizer.encode(text, add_special_tokens=False) + [tokenizer.eos_token_id]
    return {"ids": ids, "len": len(ids)}

for split_name in ["train", "val"]:
    filename = f"ft_{split_name}.bin"
    if os.path.exists(filename):
        continue
    tokenized = ds[split_name].map(
        process, remove_columns=ds[split_name].column_names,
        desc=f"tokenizing {split_name}", num_proc=2,
    )
    arr_len = np.sum(tokenized["len"], dtype=np.uint64)
    arr = np.memmap(filename, dtype=np.uint16, mode="w+", shape=(arr_len,))
    idx = 0
    for row in tqdm(tokenized, desc=f"writing {filename}"):
        ids = np.array(row["ids"], dtype=np.uint16)
        arr[idx: idx + len(ids)] = ids
        idx += len(ids)
    arr.flush()

train_data = np.memmap("ft_train.bin", dtype=np.uint16, mode="r")
val_data = np.memmap("ft_val.bin", dtype=np.uint16, mode="r")
print(f"Fine-tune train tokens: {len(train_data):,}")
print(f"Fine-tune val tokens:   {len(val_data):,}")

config.json:   0%|          | 0.00/632 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/793 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.37M [00:00<?, ?B/s]

Vocab size: 32022


tokenizing train (num_proc=2):   0%|          | 0/274514 [00:00<?, ? examples/s]

writing ft_train.bin:   0%|          | 0/274514 [00:00<?, ?it/s]

tokenizing val (num_proc=2):   0%|          | 0/5603 [00:00<?, ? examples/s]

writing ft_val.bin:   0%|          | 0/5603 [00:00<?, ?it/s]

Fine-tune train tokens: 118,758,008
Fine-tune val tokens:   2,397,235


### 5. Model architecture (identical to `base100m.ipynb`, needed to reload the weights)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
from dataclasses import dataclass

class LayerNorm(nn.Module):
    def __init__(self, ndim, bias):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(ndim))
        self.bias = nn.Parameter(torch.zeros(ndim)) if bias else None
    def forward(self, x):
        return F.layer_norm(x, self.weight.shape, self.weight, self.bias, 1e-5)

class CausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd, bias=config.bias)
        self.c_proj = nn.Linear(config.n_embd, config.n_embd, bias=config.bias)
        self.attn_dropout = nn.Dropout(config.dropout)
        self.resid_dropout = nn.Dropout(config.dropout)
        self.n_head = config.n_head
        self.n_embd = config.n_embd
        self.flash = hasattr(torch.nn.functional, "scaled_dot_product_attention")
        if not self.flash:
            self.register_buffer("bias", torch.tril(torch.ones(config.block_size, config.block_size))
                                       .view(1, 1, config.block_size, config.block_size))

    def forward(self, x):
        B, T, C = x.size()
        q, k, v = self.c_attn(x).split(self.n_embd, dim=2)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)

        if self.flash:
            y = F.scaled_dot_product_attention(q, k, v, attn_mask=None, dropout_p=self.attn_dropout.p if self.training else 0.0, is_causal=True)
        else:
            att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(k.size(-1)))
            att = att.masked_fill(self.bias[:, :, :T, :T] == 0, float("-inf"))
            att = F.softmax(att, dim=-1)
            att = self.attn_dropout(att)
            y = att @ v

        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.resid_dropout(self.c_proj(y))
        return y

class MLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.c_fc = nn.Linear(config.n_embd, 4 * config.n_embd, bias=config.bias)
        self.gelu = nn.GELU()
        self.c_proj = nn.Linear(4 * config.n_embd, config.n_embd, bias=config.bias)
        self.dropout = nn.Dropout(config.dropout)
    def forward(self, x):
        return self.dropout(self.c_proj(self.gelu(self.c_fc(x))))

class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln1 = LayerNorm(config.n_embd, config.bias)
        self.attn = CausalSelfAttention(config)
        self.ln2 = LayerNorm(config.n_embd, config.bias)
        self.mlp = MLP(config)
    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x

@dataclass
class GPTConfig:
    block_size: int = 512
    vocab_size: int = 32256
    n_layer: int = 10
    n_head: int = 12
    n_embd: int = 768
    dropout: float = 0.1
    bias: bool = True

class GPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.transformer = nn.ModuleDict(dict(
            wte=nn.Embedding(config.vocab_size, config.n_embd),
            wpe=nn.Embedding(config.block_size, config.n_embd),
            drop=nn.Dropout(config.dropout),
            h=nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
            ln_f=LayerNorm(config.n_embd, config.bias),
        ))
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        self.transformer.wte.weight = self.lm_head.weight

        self.apply(self._init_weights)
        for pn, p in self.named_parameters():
            if pn.endswith("c_proj.weight"):
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * config.n_layer))

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        dev = idx.device
        b, t = idx.size()
        assert t <= self.config.block_size, f"sequence length {t} exceeds block_size {self.config.block_size}"
        pos = torch.arange(0, t, dtype=torch.long, device=dev)

        tok_emb = self.transformer.wte(idx)
        pos_emb = self.transformer.wpe(pos)
        x = self.transformer.drop(tok_emb + pos_emb)
        for block in self.transformer.h:
            x = block(x)
        x = self.transformer.ln_f(x)

        if targets is not None:
            logits = self.lm_head(x)
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=-1)
            return logits, loss
        else:
            logits = self.lm_head(x[:, [-1], :])
            return logits, None

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None, eos_token_id=None):
        for _ in range(max_new_tokens):
            idx_cond = idx if idx.size(1) <= self.config.block_size else idx[:, -self.config.block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float("Inf")
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
            if eos_token_id is not None and idx.size(0) == 1 and idx_next.item() == eos_token_id:
                break
        return idx

### 6. Load your pretrained weights from Hugging Face

In [ ]:
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file
import json as _json

device = "cuda" if torch.cuda.is_available() else "cpu"

config_path = hf_hub_download(repo_id=HF_BASE_REPO, filename="config.json", token=HF_TOKEN or None)
weights_path = hf_hub_download(repo_id=HF_BASE_REPO, filename="model.safetensors", token=HF_TOKEN or None)

with open(config_path) as f:
    cfg_dict = _json.load(f)

config = GPTConfig(**cfg_dict)
model = GPT(config)

state_dict = load_file(weights_path)
model.load_state_dict(state_dict)
model.to(device)

# Full-parameter fine-tuning: every weight stays trainable (this is the default -
# stated explicitly so nothing is accidentally frozen, e.g. by earlier experiments with LoRA/freezing)
for p in model.parameters():
    p.requires_grad = True

n_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f"Loaded base model from {HF_BASE_REPO} - {n_params:.2f}M params, all trainable")

config.json:   0%|          | 0.00/130 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  482MB            

model.safetensors: downloading bytes:           |  0.00B            

Loaded base model from Ananda100/pocketcoder100M-distilled - 95.87M params, all trainable


### 7. Fine-tuning hyperparameters

In [ ]:
from contextlib import nullcontext

batch_size = 16
block_size = config.block_size
gradient_accumulation_steps = 16
tokens_per_step = batch_size * block_size * gradient_accumulation_steps

num_epochs = 3                       # fine-tuning usually wants several passes over a smaller dataset
target_tokens = len(train_data) * num_epochs
max_iters = max(1, int(target_tokens / tokens_per_step))
total_tokens_trained = tokens_per_step * max_iters

learning_rate = 5e-5                 # lower than the 3e-4 used for pretraining - standard for fine-tuning
warmup_steps = max(1, min(100, max_iters // 10))
min_lr = 5e-6
eval_iters = 50

device_type = "cuda" if "cuda" in device else "cpu"
dtype = "bfloat16" if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else "float16"
ptdtype = {"float32": torch.float32, "bfloat16": torch.bfloat16, "float16": torch.float16}[dtype]
ctx = nullcontext() if device_type == "cpu" else torch.amp.autocast(device_type=device_type, dtype=ptdtype)

torch.manual_seed(42)
print(f"Targeting fine-tune tokens: {total_tokens_trained:,}")
print(f"Total optimizer steps: {max_iters:,}")

Targeting fine-tune tokens: 356,253,696
Total optimizer steps: 2,718


### 8. Batching and loss estimation

In [ ]:
def get_batch(split):
    data = train_data if split == "train" else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([torch.from_numpy((data[i:i+block_size]).astype(np.int64)) for i in ix])
    y = torch.stack([torch.from_numpy((data[i+1:i+1+block_size]).astype(np.int64)) for i in ix])
    if device_type == "cuda":
        x, y = x.pin_memory().to(device, non_blocking=True), y.pin_memory().to(device, non_blocking=True)
    else:
        x, y = x.to(device), y.to(device)
    return x, y

def estimate_loss(model):
    out = {}
    model.eval()
    with torch.inference_mode():
        for split in ["train", "val"]:
            losses = torch.zeros(eval_iters)
            for k in range(eval_iters):
                X, Y = get_batch(split)
                with ctx:
                    _, loss = model(X, Y)
                losses[k] = loss.item()
            out[split] = losses.mean()
    model.train()
    return out

### 9. Optimizer and scheduler

In [ ]:
from torch.optim.lr_scheduler import LinearLR, SequentialLR, CosineAnnealingLR

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, betas=(0.9, 0.95), weight_decay=0.1, eps=1e-9)
scheduler_warmup = LinearLR(optimizer, total_iters=warmup_steps)
scheduler_decay = CosineAnnealingLR(optimizer, T_max=max(1, max_iters - warmup_steps), eta_min=min_lr)
scheduler = SequentialLR(optimizer, schedulers=[scheduler_warmup, scheduler_decay], milestones=[warmup_steps])
scaler = torch.amp.GradScaler(device_type, enabled=(dtype == "float16"))

### 10. Fine-tune (full parameter)

In [ ]:
best_val_loss = float("inf")
best_model_params_path = "best_finetuned_model.pt"
train_loss_list, validation_loss_list = [], []

model.train()
total_loop_iters = max_iters * gradient_accumulation_steps

for step in tqdm(range(total_loop_iters)):
    if step % (eval_iters * gradient_accumulation_steps) == 0 and step != 0:
        losses = estimate_loss(model)
        optimizer_step = step // gradient_accumulation_steps
        print(f"Step {optimizer_step}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")
        train_loss_list.append(losses["train"])
        validation_loss_list.append(losses["val"])
        if losses["val"] < best_val_loss:
            best_val_loss = losses["val"]
            torch.save(model.state_dict(), best_model_params_path)

    X, y = get_batch("train")
    with ctx:
        _, loss = model(X, y)
        loss = loss / gradient_accumulation_steps
        scaler.scale(loss).backward()

    if ((step + 1) % gradient_accumulation_steps == 0) or (step + 1 == total_loop_iters):
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.5)
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad(set_to_none=True)
        scheduler.step()

# Always keep a final checkpoint too, even if val loss never improved on the last logged eval
torch.save(model.state_dict(), "last_finetuned_model.pt")
if best_val_loss == float("inf"):
    best_val_loss = estimate_loss(model)["val"].item()
    torch.save(model.state_dict(), best_model_params_path)

print("--- Fine-tuning complete ---")
print(f"Total tokens processed: {total_tokens_trained:,}")
print(f"Best val loss: {best_val_loss:.4f}")

  0%|          | 0/43488 [00:00<?, ?it/s]

Step 50: train loss 1.2791, val loss 1.3296
Step 100: train loss 1.1976, val loss 1.2407
Step 150: train loss 1.1606, val loss 1.1893
Step 200: train loss 1.1095, val loss 1.1183
Step 250: train loss 1.0843, val loss 1.1209
Step 300: train loss 1.0733, val loss 1.1226
Step 350: train loss 1.0472, val loss 1.1121
Step 400: train loss 1.0541, val loss 1.0749
Step 450: train loss 1.0563, val loss 1.0726
Step 500: train loss 1.0042, val loss 1.0632
Step 550: train loss 1.0081, val loss 1.0432
Step 600: train loss 1.0296, val loss 1.0485
Step 650: train loss 1.0153, val loss 1.0079
Step 700: train loss 1.0012, val loss 1.0220
Step 750: train loss 0.9921, val loss 1.0059
Step 800: train loss 0.9954, val loss 1.0283
Step 850: train loss 0.9706, val loss 1.0248
Step 900: train loss 0.9613, val loss 1.0061
Step 950: train loss 0.9812, val loss 0.9959
Step 1000: train loss 0.9704, val loss 1.0241
Step 1050: train loss 0.9715, val loss 1.0001
Step 1100: train loss 0.9441, val loss 0.9955
Step 115

### 11. Test — does the fine-tuned model actually solve problems?

Loads the best checkpoint, generates a solution for each held-out problem, extracts the code, and runs the real `assert` tests from your dataset against it. Reports **pass@1**: the fraction of problems where every test passes.

In [ ]:
import torch

# ============================================================
# 1. Load the model checkpoint you want to test
# ============================================================
model.load_state_dict(torch.load(best_model_params_path, map_location=device))
model.eval()

# ============================================================
# 2. Core generation function
# ============================================================
def generate_solution(problem, max_new_tokens=256, temperature=0.2, top_k=50):
    prompt = f"### Problem\n{problem}\n\n### Solution\n```python\n"
    ids = tokenizer.encode(prompt, add_special_tokens=False)
    idx = torch.tensor([ids], dtype=torch.long, device=device)

    with torch.no_grad():
        out = model.generate(
            idx,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_k=top_k,
            eos_token_id=tokenizer.eos_token_id,
        )

    generated = tokenizer.decode(out[0].tolist())
    completion = generated[len(prompt):]
    if "```" in completion:
        completion = completion.split("```")[0]
    return completion.strip()


# ============================================================
# 3. Test a single prompt (clean print, no duplicate echo)
# ============================================================
def ask(problem, max_new_tokens=256, temperature=0.2, top_k=50, n_samples=1):
    print("PROMPT:")
    print(problem)
    for i in range(n_samples):
        code = generate_solution(problem, max_new_tokens, temperature, top_k)
        label = f"\nGENERATED CODE (sample {i+1}/{n_samples}):" if n_samples > 1 else "\nGENERATED CODE:"
        print(label)
        print(code)
    print()


# ============================================================
# 4. Test a batch of prompts in one go
# ============================================================
def ask_many(problems, max_new_tokens=256, temperature=0.2, top_k=50):
    for p in problems:
        ask(p, max_new_tokens, temperature, top_k)
        print("=" * 60)


# ============================================================
# 5. Optional: test a prompt AND check it against your own asserts
#    (reuses the timeout-protected execution pattern from the eval cell)
# ============================================================
import signal

class GenTimeout(Exception):
    pass

def _handler(signum, frame):
    raise GenTimeout()

def ask_and_verify(problem, tests, max_new_tokens=256, temperature=0.2, top_k=50, timeout_s=5):
    code = generate_solution(problem, max_new_tokens, temperature, top_k)
    print("PROMPT:")
    print(problem)
    print("\nGENERATED CODE:")
    print(code)

    namespace = {}
    passed, total = 0, len(tests)
    try:
        signal.signal(signal.SIGALRM, _handler)
        signal.alarm(timeout_s)
        exec(code, namespace)
        for t in tests:
            try:
                exec(t, namespace)
                passed += 1
            except Exception as e:
                print(f"  FAILED: {t}  ({e})")
        signal.alarm(0)
    except GenTimeout:
        print("  TIMEOUT: generated code did not finish executing")
    except Exception as e:
        print(f"  ERROR running generated code: {e}")
        signal.alarm(0)

    print(f"\nResult: {passed}/{total} tests passed")
    print()
    return passed, total


# ============================================================
# USAGE EXAMPLES — uncomment / edit as needed
# ============================================================

# -- Single prompt --
ask("Write a function that returns all prime numbers less than n.")

# -- Same prompt, multiple samples (useful since temperature=0.2 is near-deterministic,
#    bump temperature up to see variation) --
# ask("Write a function that checks if a number is prime.", temperature=0.8, n_samples=3)

# -- Batch of prompts --
ask_many([ "Write a function that performs binary search.",
    "Write a function that validates an email address.",
    "Write a function that reads a JSON file.",
    "Write a function that writes data to a CSV file.",
    "Write a function that sends an HTTP GET request.",
    "Write a function that downloads a file from a URL.",
    "Write a function that parses command line arguments using argparse.",
    "Write a function that computes the SHA256 hash of a file.",
    "Write a function that compresses a file into a ZIP archive.",
    "Write a function that extracts a ZIP archive.",
    "Write a function that finds all files with a given extension.",
    "Write a function that recursively lists all files in a directory.",
    "Write a function that copies a directory.",
    "Write a function that removes duplicate lines from a text file.",
    "Write a function that counts the number of lines in a file.",
    "Write a function that loads a CSV file into a pandas DataFrame.",
    "Write a function that removes missing values from a DataFrame.",
    "Write a function that groups a DataFrame by a column.",
    "Write a function that computes the mean of each column.",
    "Write a function that plots a histogram using matplotlib.",
    "Write a function that trains a linear regression model using scikit-learn.",
    "Write a function that splits a dataset into training and testing sets.",
    "Write a function that computes classification accuracy.",
    "Write a function that tokenizes a sentence.",
    "Write a function that removes punctuation from a string.",
    "Write a function that converts text to lowercase.",
    "Write a function that counts the frequency of each word.",
    "Write a function that finds the longest common prefix of a list of strings.",
    "Write a function that implements an LRU cache.",
    "Write a function that implements breadth-first search on a graph.",
    "Write a function that implements depth-first search on a graph.",
    "Write a function that finds the shortest path using Dijkstra's algorithm.",
    "Write a function that detects a cycle in a graph.",
    "Write a function that performs topological sorting.",
    "Write a function that implements a queue using two stacks.",
    "Write a function that implements a stack using two queues.",
    "Write a function that checks whether parentheses are balanced.",
    "Write a function that evaluates a postfix expression.",
    "Write a function that merges overlapping intervals.",
    "Write a function that finds the k largest elements in a list.",
    "Write a function that returns the median of two sorted arrays.",
    "Write a function that implements binary exponentiation.",
    "Write a function that computes the greatest common divisor.",
    "Write a function that computes the least common multiple.",
    "Write a function that generates all permutations of a list.",
    "Write a function that generates all subsets of a list.",
    "Write a function that solves the Two Sum problem.",
    "Write a function that rotates a matrix by 90 degrees.",
    "Write a function that transposes a matrix.",
    "Write a function that checks whether a Sudoku board is valid."])

# -- Prompt + your own correctness check --
# ask_and_verify(
#     "Write a function called is_even that returns True if a number is even.",
#     tests=[
#         "assert is_even(4) == True",
#         "assert is_even(7) == False",
#         "assert is_even(0) == True",
#     ]
# )

PROMPT:
Write a function that returns all prime numbers less than n.

GENERATED CODE:
def is_prime(n):
    if n <= 1:
        return False
    for i in range(2, int(n**0.5) + 1):
        if n % i == 0:
            return False
    return True

def find_primes(n):
    primes = []
    for num in range(2, n + 1):
        if is_prime(num):
            primes.append(num)
    return primes

PROMPT:
Write a function that performs binary search.

GENERATED CODE:
def binary_search(arr, target):
    left, right = 0, len(arr) - 1
    while left <= right:
        mid = (left + right) // 2
        if arr[mid] == target:
            return mid
        elif arr[mid] < target:
            left = mid + 1
        else:
            right = mid - 1
    return -1

PROMPT:
Write a function that validates an email address.

GENERATED CODE:
import re

def validate_email(email):
    pattern = r'^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$'
    return re.match(pattern, email) is not None

PROMPT:
Write a fu

In [ ]:
HF_FT_REPO = "Ananda100/pocketcoder-sft"

## 12. Push the fine-tuned model to the Hugging Face Hub

In [ ]:
from safetensors.torch import save_file
from huggingface_hub import HfApi, create_repo
from dataclasses import asdict


model.load_state_dict(torch.load(best_model_params_path, map_location=device))
model.eval()

os.makedirs("hf_ft_upload", exist_ok=True)

state_dict = {k: v.contiguous().clone() for k, v in model.state_dict().items()}
save_file(state_dict, "hf_ft_upload/model.safetensors")

with open("hf_ft_upload/config.json", "w") as f:
    json.dump(asdict(config), f, indent=2)

tokenizer.save_pretrained("hf_ft_upload")

readme = f"""---
license: apache-2.0
tags:
  - nanoGPT
  - code-generation
  - fine-tuned
base_model: {HF_BASE_REPO}
---

# {HF_FT_REPO.split('/')[-1]}

Full-parameter fine-tune of [`{HF_BASE_REPO}`](https://huggingface.co/{HF_BASE_REPO})
on a custom Python coding-problem dataset, using the `deepseek-ai/deepseek-coder-6.7b-base` tokenizer.

- Fine-tune tokens processed: {total_tokens_trained:,}
- Best validation loss: {best_val_loss:.4f}
- Held-out pass@1: {full_pass_rate:.1%}
- Block size: {config.block_size}
- Vocab size: {config.vocab_size}

## Loading

This is a **custom architecture**, not a native `transformers` model, so `AutoModel.from_pretrained`
won't work out of the box:

```python
import json, torch
from safetensors.torch import load_file

with open("config.json") as f:
    cfg = json.load(f)

config = GPTConfig(**cfg)
model = GPT(config)
state_dict = load_file("model.safetensors")
model.load_state_dict(state_dict)
model.eval()
```
"""
with open("hf_ft_upload/README.md", "w") as f:
    f.write(readme)

create_repo(HF_FT_REPO, token=HF_TOKEN, exist_ok=True)
api = HfApi()
api.upload_folder(folder_path="hf_ft_upload", repo_id=HF_FT_REPO, token=HF_TOKEN)
print(f"Pushed to https://huggingface.co/{HF_FT_REPO}")

Pushed to https://huggingface.co/Ananda100/100m-sftd-python
